# 01 micrograd：手写标量自动求导

目标：不看参考代码，实现一个 `Value` 类，支持加、乘、幂、ReLU，并能对任意表达式图做反向传播；再用它搭一个两层 MLP 训练到 100% 准确。

配套：视频 Zero to Hero 第 1 集"The spelled-out intro to neural networks and backpropagation"；参考实现在 ../../reference/micrograd/micrograd/engine.py，卡住超过 30 分钟再看。

规则：先把 TODO 填完，再运行测试格；测试格用 PyTorch 作标准答案，全部 assert 通过即完成。


In [ ]:
import math, random
random.seed(0)

class Value:
    """一个标量，记录它由哪些 Value 经什么运算得到，以便反向传播。"""
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None     # 由产生它的运算设置：把 self.grad 分配给 children
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            # TODO: 加法的局部导数是 1，把 out.grad 累加到 self.grad 和 other.grad
            raise NotImplementedError
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            # TODO: d(a*b)/da = b，d(a*b)/db = a
            raise NotImplementedError
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), '只支持常数幂'
        out = Value(self.data ** k, (self,), f'**{k}')
        def _backward():
            # TODO: d(x^k)/dx = k * x^(k-1)
            raise NotImplementedError
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'ReLU')
        def _backward():
            # TODO: 输入大于 0 时导数为 1，否则为 0
            raise NotImplementedError
        out._backward = _backward
        return out

    def backward(self):
        """从 self 出发，按拓扑序反向调用每个节点的 _backward。"""
        topo, visited = [], set()
        def build(v):
            # TODO: 深度优先，把 v 的所有祖先先加入 topo，再加入 v
            raise NotImplementedError
        build(self)
        self.grad = 1.0
        # TODO: 逆序遍历 topo，调用每个节点的 _backward()
        raise NotImplementedError

    # 下面这些靠上面四个运算组合出来，不用改
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self): return f'Value(data={self.data:.4f}, grad={self.grad:.4f})'


In [ ]:
# ===== 测试 1：和 PyTorch 对答案 =====
import torch
def check_against_torch():
    a, b = Value(-4.0), Value(2.0)
    c = a + b; d = a * b + b ** 3
    c = c + c + 1; c = c + 1 + c + (-a)
    d = d + d * 2 + (b + a).relu(); d = d + 3 * d + (b - a).relu()
    e = c - d; f = e ** 2; g = f / 2.0; g = g + 10.0 / f
    g.backward()
    ta, tb = torch.tensor(-4.0, requires_grad=True), torch.tensor(2.0, requires_grad=True)
    tc = ta + tb; td = ta * tb + tb ** 3
    tc = tc + tc + 1; tc = tc + 1 + tc + (-ta)
    td = td + td * 2 + (tb + ta).relu(); td = td + 3 * td + (tb - ta).relu()
    te = tc - td; tf = te ** 2; tg = tf / 2.0; tg = tg + 10.0 / tf
    tg.backward()
    assert abs(g.data - tg.item()) < 1e-6, ('前向值不对', g.data, tg.item())
    assert abs(a.grad - ta.grad.item()) < 1e-6, ('a 的梯度不对', a.grad, ta.grad.item())
    assert abs(b.grad - tb.grad.item()) < 1e-6, ('b 的梯度不对', b.grad, tb.grad.item())
    print('测试 1 通过：前向值 %.4f，dg/da=%.4f，dg/db=%.4f' % (g.data, a.grad, b.grad))
check_against_torch()


In [ ]:
# ===== 第二部分：用 Value 搭一个 MLP =====
class Module:
    def zero_grad(self):
        for p in self.parameters(): p.grad = 0.0
    def parameters(self): return []

class Neuron(Module):
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin
    def __call__(self, x):
        # TODO: act = sum(w_i * x_i) + b；nonlin 为真时返回 act.relu()
        raise NotImplementedError
    def parameters(self): return self.w + [self.b]

class Layer(Module):
    def __init__(self, nin, nout, **kw):
        self.neurons = [Neuron(nin, **kw) for _ in range(nout)]
    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

class MLP(Module):
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        # TODO: 建 len(nouts) 层，最后一层 nonlin=False
        raise NotImplementedError
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        return x
    def parameters(self): return [p for l in self.layers for p in l.parameters()]


In [ ]:
# ===== 测试 2：在 4 个样本上训练到全对 =====
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
model = MLP(3, [4, 4, 1])
print('参数个数', len(model.parameters()))
for step in range(100):
    preds = [model(x) for x in xs]
    loss = sum((p - y) ** 2 for p, y in zip(preds, ys))
    model.zero_grad()
    loss.backward()
    # TODO: 梯度下降，学习率 0.05：p.data -= lr * p.grad
    raise NotImplementedError
    if step % 20 == 0: print(f'step {step:3d} loss {loss.data:.4f}')
preds = [model(x).data for x in xs]
assert all((p > 0) == (y > 0) for p, y in zip(preds, ys)), ('还没全对', preds)
print('测试 2 通过：预测', [round(p, 2) for p in preds], '目标', ys)


## 完成后回答（写在这里）
1. 为什么 `_backward` 里用 `+=` 而不是 `=`？构造一个让 `=` 出错的表达式验证。
2. 拓扑排序保证了什么？如果随机顺序调用 `_backward` 会怎样？
3. `zero_grad` 不调用会发生什么？去掉它跑一次，观察 loss。
4. 这个实现和 PyTorch 的根本差别是什么？提示：标量 vs 张量、Python 循环 vs 向量化。
